In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. 쪼개진 일꾼 클러스터 세션 연결
spark = SparkSession.builder \
    .master("spark://jupyter:7077") \
    .appName("Real_Data_Analysis") \
    .config("spark.executor.memory", "1g") \
    .getOrCreate()

In [5]:
# 2. CSV 데이터 로드 (Transformation)
# 인코딩이 안 맞아서 에러가 나거나 한글이 깨지면 "CP949" 대신 "UTF-8"이나 "EUC-KR"로 바꿔보세요.
file_path = "/home/jovyan/work/01/행정안전부_착한가격업소 현황_20260630.csv"

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("encoding", "UTF-8") \
    .load(file_path)

# 3. 데이터프레임 구조와 형태 확인 (Action)
print("📌 [구조 확인] 데이터의 스키마(컬럼명과 타입)를 출력합니다:")
df.printSchema()

print(f"📌 [개수 확인] 총 행의 개수: {df.count()}개")

print("📌 [데이터 맛보기] 상위 5개 행을 보여줍니다:")
df.show(5)

📌 [구조 확인] 데이터의 스키마(컬럼명과 타입)를 출력합니다:
root
 |-- 시도: string (nullable = true)
 |-- 시군: string (nullable = true)
 |-- 업종: string (nullable = true)
 |-- 업소명: string (nullable = true)
 |-- 연락처: string (nullable = true)
 |-- 주소: string (nullable = true)
 |-- 메뉴1: string (nullable = true)
 |-- 가격1: string (nullable = true)
 |-- 메뉴2: string (nullable = true)
 |-- 가격2: string (nullable = true)
 |-- 메뉴3: string (nullable = true)
 |-- 가격3: string (nullable = true)
 |-- 메뉴4: string (nullable = true)
 |-- 가격4: string (nullable = true)

📌 [개수 확인] 총 행의 개수: 12645개
📌 [데이터 맛보기] 상위 5개 행을 보여줍니다:
+----------+------+----------+-----------+------------+--------------------------------+-----------+-----+-----------+-----+-----+-----+-----+-----+
|      시도|  시군|      업종|     업소명|      연락처|                            주소|      메뉴1|가격1|      메뉴2|가격2|메뉴3|가격3|메뉴4|가격4|
+----------+------+----------+-----------+------------+--------------------------------+-----------+-----+-----------+-----+-----+-----+-----+-----+
|

In [6]:
# 1. inferSchema 옵션을 false로 끕니다. (추론 과정을 없애서 일꾼들이 바로 쪼갤 수 있게 만듦)
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("encoding", "UTF-8") \
    .load(file_path)

# 2. 데이터를 강제로 4개의 조각(파티션)으로 강제 분할합니다.
df_repartitioned = df.repartition(4)

# 3. 개수를 세어봅니다 (Action)
print(f"📌 총 행의 개수: {df_repartitioned.count()}개")


📌 총 행의 개수: 12645개


In [9]:
from pyspark.sql.functions import col, avg, round
from pyspark.sql.types import IntegerType

# 1. 가격 데이터를 정수형으로 변환 (숫자 계산을 위해 필수)
clean_df = df.withColumn("Price", col("가격1").cast(IntegerType())).filter(col("Price").isNotNull())

# 2. 핵심: 업종을 첫 번째 기준으로 쪼개고, 그 안에서 시도별로 또 쪼개기
# rollup("업종", "시도")를 쓰면 [업종별 총평균]과 [업종 내 시도별 평균]을 한 번에 계산합니다.
result = clean_df.rollup("업종", "시도") \
                 .agg(round(avg("Price"), 0).alias("평균가격")) \
                 .orderBy("업종", "시도")

# 3. 결과 보기 (Action)
result.show(100)  # 업종이 많으므로 50개까지 넓게 출력


+------------+--------------+--------+
|        업종|          시도|평균가격|
+------------+--------------+--------+
|        NULL|          NULL| 12453.0|
|기타비요식업|          NULL| 65394.0|
|기타비요식업|강원특별자치도| 41900.0|
|기타비요식업|        경기도| 76964.0|
|기타비요식업|      경상남도| 36170.0|
|기타비요식업|      경상북도| 40143.0|
|기타비요식업|    광주광역시| 63545.0|
|기타비요식업|    대구광역시| 44571.0|
|기타비요식업|    대전광역시| 21670.0|
|기타비요식업|    부산광역시| 44058.0|
|기타비요식업|    서울특별시| 96951.0|
|기타비요식업|세종특별자치시| 33000.0|
|기타비요식업|    울산광역시|361680.0|
|기타비요식업|    인천광역시| 43800.0|
|기타비요식업|      전라남도| 20833.0|
|기타비요식업|전북특별자치도| 74625.0|
|기타비요식업|제주특별자치도| 25011.0|
|기타비요식업|      충청남도| 43373.0|
|기타비요식업|      충청북도| 33018.0|
|  기타요식업|          NULL|  3555.0|
|  기타요식업|강원특별자치도|  3820.0|
|  기타요식업|        경기도|  3821.0|
|  기타요식업|      경상남도|  4251.0|
|  기타요식업|      경상북도|  3158.0|
|  기타요식업|    광주광역시|  4400.0|
|  기타요식업|    대구광역시|  3925.0|
|  기타요식업|    대전광역시|  3041.0|
|  기타요식업|    부산광역시|  3102.0|
|  기타요식업|    서울특별시|  3754.0|
|  기타요식업|세종특별자치시|   900.0|
|  기타요식업|    울산광역시| 